# EDA - Gold Products

**Source:** `gold.products` (PostgreSQL, `marketpulse` database)

381 rows, ~2.4 MB — small enough to load everything including embeddings.

In [1]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

engine = create_engine(os.environ["POSTGRES_GOLD_CONN"])

# Products is small — load everything except the embedding array column
PRODUCTS_QUERY = """
SELECT
    product_id, brand, product_category, product_name, category_id,
    product_page_url, avg_rating, total_review_count,
    recommended_count, total_photo_count,
    rating_dist_1, rating_dist_2, rating_dist_3, rating_dist_4, rating_dist_5,
    product_name_clean, product_name_lemmas,
    rating_entropy, polarization_score, embedding_norm_name,
    revision_date
FROM gold.products
"""

df = pd.read_sql(PRODUCTS_QUERY, engine)
print(f"Loaded {len(df):,} rows | {df.memory_usage(deep=True).sum() / 1e6:.2f} MB in memory")
df.head()

Loaded 381 rows | 0.20 MB in memory


,product_id,brand,product_category,product_name,category_id,product_page_url,avg_rating,total_review_count,recommended_count,total_photo_count,...,rating_dist_2,rating_dist_3,rating_dist_4,rating_dist_5,product_name_clean,product_name_lemmas,rating_entropy,polarization_score,embedding_norm_name,revision_date
0,P517156,Caudalie,sunscreen,Dark Spot Brightening Serum & Sunscreen SPF 50...,cat60105,https://www.sephora.com/product/dark-spot-brig...,4.333334,18,15,3,...,1,2,1,13,dark spot brightening serum sunscreen spf 50 set,dark | spot | brightening | serum | sunscreen ...,1.386274,0.777778,0.999895,2026-04-05
1,P510877,caliray,hyaluronic,Sundrip Luminous Peptide Serum Bronzer,cat60018,https://www.sephora.com/product/caliray-sundri...,4.730089,226,213,55,...,2,13,17,191,sundrip luminous peptide serum bronzer,sundrip | luminous | peptide | serum | bronzer,0.866036,0.858407,0.999919,2026-04-05
2,P501003,Murad,other,Clarifying Water Gel Moisturizer with Hyaluron...,cat60097,,3.857143,21,15,1,...,0,0,0,15,clarifying water gel moisturizer hyaluronic acid,clarify | water | gel | moisturizer | hyaluron...,0.863121,1.000000,0.999731,2026-04-05
3,P482740,Shiseido,hyaluronic,Mini Urban Environment Oil-Free SPF 42 Face Su...,cat920033,https://www.sephora.com/product/shiseido-mini-...,3.942029,69,49,31,...,6,4,15,36,mini urban environment oil free spf 42 face su...,mini | urban | environment | oil | free | spf ...,1.873301,0.637681,0.999723,2026-04-05
4,P470023,Dr. Dennis Gross Skincare,sunscreen,All-Physical Lightweight Wrinkle Defense Broad...,cat920033,https://www.sephora.com/product/dr-dennis-gros...,4.302387,377,319,20,...,23,26,86,228,physical lightweight wrinkle defense broad spe...,physical | lightweight | wrinkle | defense | b...,1.613821,0.641910,0.999880,2026-04-05


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 381 entries, 0 to 380
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_id           381 non-null    str    
 1   brand                381 non-null    str    
 2   product_category     381 non-null    str    
 3   product_name         381 non-null    str    
 4   category_id          381 non-null    str    
 5   product_page_url     381 non-null    str    
 6   avg_rating           381 non-null    float64
 7   total_review_count   381 non-null    int64  
 8   recommended_count    381 non-null    int64  
 9   total_photo_count    381 non-null    int64  
 10  rating_dist_1        381 non-null    int64  
 11  rating_dist_2        381 non-null    int64  
 12  rating_dist_3        381 non-null    int64  
 13  rating_dist_4        381 non-null    int64  
 14  rating_dist_5        381 non-null    int64  
 15  product_name_clean   381 non-null    str    
 16  p

In [3]:
df.describe()

,avg_rating,total_review_count,recommended_count,total_photo_count,rating_dist_1,rating_dist_2,rating_dist_3,rating_dist_4,rating_dist_5,rating_entropy,polarization_score,embedding_norm_name
count,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000,381.000000
mean,4.377051,568.721785,467.496063,157.275591,34.013123,26.314961,36.734908,92.766404,378.892388,1.288986,0.758873,0.999989
std,0.425987,877.942095,708.174921,235.403361,74.806504,53.799731,64.637618,152.810548,562.512415,0.435898,0.108118,0.000176
min,1.604651,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.333333,0.999682
25%,4.218000,107.000000,90.000000,24.000000,2.000000,2.000000,4.000000,12.000000,80.000000,0.963144,0.680486,0.999828
50%,4.465116,262.000000,241.000000,64.000000,9.000000,7.000000,13.000000,34.000000,196.000000,1.323533,0.761714,0.999988
75%,4.652747,677.000000,530.000000,178.000000,32.000000,23.000000,37.000000,105.000000,438.000000,1.598336,0.835484,1.000142
max,5.000000,9013.000000,7503.000000,1474.000000,826.000000,441.000000,488.000000,1336.000000,5922.000000,2.207720,1.000000,1.000311


## (Optional) Load product name embeddings

In [4]:
emb_df = pd.read_sql("SELECT product_id, product_name_embedding FROM gold.products", engine)
product_name_emb = np.array(emb_df["product_name_embedding"].tolist(), dtype=np.float32)
print(f"product_name_emb: {product_name_emb.shape} | {product_name_emb.nbytes / 1e6:.1f} MB")

product_name_emb: (381, 1024) | 1.6 MB
